In [1]:
# Task 4: Forecasting Access and Usage (2025-2027)
# Ethiopia Financial Inclusion — Selam Analytics


In [2]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

from impact_model import event_effect_at_time, total_effect_at_date

plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', None)


In [ ]:
# load enriched data
df = pd.read_csv('../data/processed/ethiopia_fi_enriched.csv', parse_dates=['observation_date'])
obs = df[df['record_type'] == 'observation'].copy()
events_df = df[df['record_type'] == 'event'].copy()
links_df = df[df['record_type'] == 'impact_link'].copy()

obs['year'] = obs['observation_date'].dt.year
obs['year_num'] = obs['observation_date'].dt.year + (obs['observation_date'].dt.month - 1) / 12


In [5]:
#  Trend regression - Access (ACC_OWNERSHIP) 
def fit_trend(indicator_code, obs_df):
    """Fit OLS trend line: value ~ year, with confidence interval support."""
    data = obs_df[obs_df['indicator_code'] == indicator_code].dropna(subset=['value_numeric']).sort_values('year_num')
    X = sm.add_constant(data['year_num'])
    y = data['value_numeric']
    model = sm.OLS(y, X).fit()
    return model, data

access_model, access_data = fit_trend('ACC_OWNERSHIP', obs)
print(access_model.summary())

                            OLS Regression Results                            
Dep. Variable:          value_numeric   R-squared:                       0.685
Model:                            OLS   Adj. R-squared:                  0.607
Method:                 Least Squares   F-statistic:                     8.712
Date:                Tue, 21 Jul 2026   Prob (F-statistic):             0.0419
Time:                        12:01:40   Log-Likelihood:                -19.471
No. Observations:                   6   AIC:                             42.94
Df Residuals:                       4   BIC:                             42.52
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -5691.5194   1942.083     -2.931      0.0

/home/sumeya/Documents/ai project/10x academi/ethiopia-fi-forecast/venv/lib/python3.12/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [8]:
print(sorted(obs['indicator_code'].dropna().unique()))

['ACC_4G_COV', 'ACC_FAYDA', 'ACC_MM_ACCOUNT', 'ACC_MOBILE_PEN', 'ACC_OWNERSHIP', 'AFF_DATA_INCOME', 'GEN_GAP_ACC', 'GEN_GAP_MOBILE', 'GEN_MM_SHARE', 'USG_ACTIVE_RATE', 'USG_ATM_COUNT', 'USG_ATM_VALUE', 'USG_CROSSOVER', 'USG_MPESA_ACTIVE', 'USG_MPESA_USERS', 'USG_P2P_COUNT', 'USG_P2P_VALUE', 'USG_TELEBIRR_USERS', 'USG_TELEBIRR_VALUE']


In [9]:
obs[obs['indicator_code'] == 'USG_ACTIVE_RATE'][['observation_date', 'value_numeric', 'source_name', 'original_text']]

,observation_date,value_numeric,source_name,original_text
24,2024-12-31,66.0,Calculated,7.1M / 10.8M = 66%


In [11]:
new_usage_obs = pd.DataFrame([{
    "record_id": "OBS_NEW_TASK4_001",
    "record_type": "observation",
    "pillar": "usage",
    "indicator": "Made or received a digital payment (% age 15+)",
    "indicator_code": "USG_DIGITAL_PAYMENT",
    "value_numeric": 35.0,
    "observation_date": "2024-06-01",
    "category": None,
    "source_type": "survey",
    "source_name": "Global Findex 2024",
    "source_url": "https://www.worldbank.org/globalfindex",
    "original_text": "Made or received digital payment: ~35%",
    "confidence": "high",
    "parent_id": None, "related_indicator": None, "impact_direction": None,
    "impact_magnitude": None, "lag_months": None, "evidence_basis": None,
    "collected_by": "Sumeya Hassen",
    "collection_date": "2026-07-20",
    "notes": "Anchor point for Usage forecasting.",
}])

df = pd.concat([df, new_usage_obs], ignore_index=True, sort=False)
df['observation_date'] = pd.to_datetime(df['observation_date'])   # fixes the dtype issue
df.to_csv('../data/processed/ethiopia_fi_enriched.csv', index=False)

obs = df[df['record_type'] == 'observation'].copy()
obs['year'] = obs['observation_date'].dt.year
obs['year_num'] = obs['observation_date'].dt.year + (obs['observation_date'].dt.month - 1) / 12
print("Anchor added and saved.")

Anchor added and saved.


In [12]:
def fit_trend(indicator_code, obs_df):
    data = obs_df[obs_df['indicator_code'] == indicator_code].dropna(subset=['value_numeric']).sort_values('year_num')
    X = sm.add_constant(data['year_num'])
    y = data['value_numeric']
    model = sm.OLS(y, X).fit()
    return model, data

# Access: real OLS trend (5 Findex points)
access_model, access_data = fit_trend('ACC_OWNERSHIP', obs)

# Growth-rate driver: use ACC_MM_ACCOUNT's historical CAGR as Usage's growth proxy
mm_data = obs[obs['indicator_code'] == 'ACC_MM_ACCOUNT'].dropna(subset=['value_numeric']).sort_values('year_num')
print(mm_data[['observation_date', 'value_numeric', 'year_num']])

years_span = mm_data['year_num'].iloc[-1] - mm_data['year_num'].iloc[0]
cagr = (mm_data['value_numeric'].iloc[-1] / mm_data['value_numeric'].iloc[0]) ** (1/years_span) - 1
print(f"\nMobile money CAGR (used as Usage growth proxy): {cagr:.2%} per year")

# Project Usage forward from the 35% anchor
usage_anchor = 35.0
usage_anchor_year = 2024.5
target_years = [2025.0, 2026.0, 2027.0]

usage_projection = pd.DataFrame({
    'year': target_years,
    'mean': [usage_anchor * (1 + cagr) ** (y - usage_anchor_year) for y in target_years]
})
usage_projection['mean_ci_lower'] = usage_projection['mean'] * 0.85
usage_projection['mean_ci_upper'] = usage_projection['mean'] * 1.15

print("\nUsage projection:")
print(usage_projection.round(2))

  observation_date  value_numeric     year_num
6       2021-12-31           4.70  2021.916667
7       2024-11-29           9.45  2024.833333

Mobile money CAGR (used as Usage growth proxy): 27.06% per year

Usage projection:
     year   mean  mean_ci_lower  mean_ci_upper
0  2025.0  39.45          33.53          45.37
1  2026.0  50.13          42.61          57.65
2  2027.0  63.69          54.14          73.24


In [13]:
# Use pp/year growth instead of compound % growth (more realistic at a higher base)
mm_pp_change = mm_data['value_numeric'].iloc[-1] - mm_data['value_numeric'].iloc[0]
mm_years_span = mm_data['year_num'].iloc[-1] - mm_data['year_num'].iloc[0]
pp_growth_per_year = mm_pp_change / mm_years_span
print(f"Mobile money growth rate: {pp_growth_per_year:.2f} percentage points/year")

usage_anchor = 35.0
usage_anchor_year = 2024.5
target_years = [2025.0, 2026.0, 2027.0]

usage_projection = pd.DataFrame({
    'year': target_years,
    'mean': [usage_anchor + pp_growth_per_year * (y - usage_anchor_year) for y in target_years]
})
usage_projection['mean_ci_lower'] = usage_projection['mean'] - 3.0   # +/- 3pp band, widen with horizon
usage_projection['mean_ci_upper'] = usage_projection['mean'] + 3.0
usage_projection['mean_ci_lower'] = usage_projection.apply(
    lambda r: r['mean_ci_lower'] - 2 * (r['year'] - 2025), axis=1)
usage_projection['mean_ci_upper'] = usage_projection.apply(
    lambda r: r['mean_ci_upper'] + 2 * (r['year'] - 2025), axis=1)

print("\nUsage projection (pp/year method):")
print(usage_projection.round(2))

Mobile money growth rate: 1.63 percentage points/year

Usage projection (pp/year method):
     year   mean  mean_ci_lower  mean_ci_upper
0  2025.0  35.81          32.81          38.81
1  2026.0  37.44          32.44          42.44
2  2027.0  39.07          32.07          46.07


In [14]:
def event_augmented_forecast(baseline_df, indicator_code, events_df, links_df):
    """Add cumulative modeled event effects on top of the baseline projection."""
    augmented = baseline_df.copy()
    event_effects = []
    for year in augmented['year']:
        target_date = pd.Timestamp(f'{int(year)}-01-01')
        effect = total_effect_at_date(target_date, indicator_code, events_df, links_df)
        event_effects.append(effect)
    augmented['event_effect'] = event_effects
    augmented['mean_with_events'] = augmented['mean'] + augmented['event_effect']
    augmented['ci_lower_with_events'] = augmented['mean_ci_lower'] + augmented['event_effect']
    augmented['ci_upper_with_events'] = augmented['mean_ci_upper'] + augmented['event_effect']
    return augmented

events_df = df[df['record_type'] == 'event'].copy()
links_df = df[df['record_type'] == 'impact_link'].copy()

# Access baseline (OLS trend + CI)
def forecast_trend(model, target_years):
    X_future = sm.add_constant(pd.DataFrame({'year_num': target_years}), has_constant='add')
    pred = model.get_prediction(X_future)
    summary = pred.summary_frame(alpha=0.20)
    summary['year'] = target_years
    return summary

target_years = [2025.0, 2026.0, 2027.0]
access_baseline = forecast_trend(access_model, target_years)

access_augmented = event_augmented_forecast(access_baseline, 'ACC_OWNERSHIP', events_df, links_df)
usage_augmented = event_augmented_forecast(usage_projection, 'USG_DIGITAL_PAYMENT', events_df, links_df)

print("Access — trend vs event-augmented:")
print(access_augmented[['year', 'mean', 'event_effect', 'mean_with_events']].round(2))
print("\nUsage — trend vs event-augmented:")
print(usage_augmented[['year', 'mean', 'event_effect', 'mean_with_events']].round(2))

Access — trend vs event-augmented:
     year   mean  event_effect  mean_with_events
0  2025.0  53.24          5.00             58.24
1  2026.0  56.07          5.00             61.07
2  2027.0  58.91          7.38             66.29

Usage — trend vs event-augmented:
     year   mean  event_effect  mean_with_events
0  2025.0  35.81           0.0             35.81
1  2026.0  37.44           0.0             37.44
2  2027.0  39.07           0.0             39.07


In [15]:
def build_scenarios(augmented_df):
    scenarios = augmented_df.copy()
    scenarios['pessimistic'] = scenarios['ci_lower_with_events'] * 0.9 + scenarios['event_effect'] * 0.1
    scenarios['base'] = scenarios['mean_with_events']
    scenarios['optimistic'] = scenarios['ci_upper_with_events'] * 1.05
    return scenarios[['year', 'pessimistic', 'base', 'optimistic']]

access_scenarios = build_scenarios(access_augmented)
usage_scenarios = build_scenarios(usage_augmented)

print("Access scenarios (%):")
print(access_scenarios.round(1))
print("\nUsage scenarios (%):")
print(usage_scenarios.round(1))

Access scenarios (%):
     year  pessimistic  base  optimistic
0  2025.0         45.6  58.2        69.6
1  2026.0         47.1  61.1        73.9
2  2027.0         50.9  66.3        80.7

Usage scenarios (%):
     year  pessimistic  base  optimistic
0  2025.0         29.5  35.8        40.8
1  2026.0         29.2  37.4        44.6
2  2027.0         28.9  39.1        48.4


In [16]:
def fit_trend(indicator_code, obs_df, min_year=None):
    data = obs_df[obs_df['indicator_code'] == indicator_code].dropna(subset=['value_numeric']).sort_values('year_num')
    if min_year:
        data = data[data['year_num'] >= min_year]
    X = sm.add_constant(data['year_num'])
    y = data['value_numeric']
    model = sm.OLS(y, X).fit()
    return model, data

# Use only 2017-2024 (last 3 Findex rounds) to reflect the deceleration, not the early high-growth years
access_model, access_data = fit_trend('ACC_OWNERSHIP', obs, min_year=2017)
print(access_model.summary())

                            OLS Regression Results                            
Dep. Variable:          value_numeric   R-squared:                       0.338
Model:                            OLS   Adj. R-squared:                  0.117
Method:                 Least Squares   F-statistic:                     1.532
Date:                Tue, 21 Jul 2026   Prob (F-statistic):              0.304
Time:                        12:11:46   Log-Likelihood:                -16.439
No. Observations:                   5   AIC:                             36.88
Df Residuals:                       3   BIC:                             36.10
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -4204.9182   3433.054     -1.225      0.3

/home/sumeya/Documents/ai project/10x academi/ethiopia-fi-forecast/venv/lib/python3.12/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 5 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "
